#   Traffic Count using Yolo12 and Faster R-CNN

## YOLO12

## Installing Libraries

In [29]:
!pip install ultralytics opencv-python-headless norfair

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 3.9 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.9 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110458 sha256=68887a788832996c60f2d3461f37587f34b1fab8037ac9ce8b297cfd1d1bb6c9
  Stored in directory: /root/.cache/pip/wheels/12/dc/3c/e12983eac132d00f82a20c6cbe7b42ce6e96190ef8fa2d15e1
Successfully built filterpy


## Importing Libraries

In [6]:
import cv2
import numpy as np
import os
from ultralytics import YOLO
from collections import defaultdict
import math


## Defining parameters

In [15]:
# ==== CONFIGURATION ====
VIDEO_PATH = "Your Video Path"  # change this
SPEED_LIMIT_KMPH = 30
FRAME_RATE = 50  # assumed FPS
PIXELS_PER_FOOT_X = 8
PIXELS_PER_FOOT_Y = 20
COUNT_LINE_Y = 500  # y-position of virtual counting line


## Model Selections and Defining Classes

In [16]:
# ==== INITIALIZE ====
model = YOLO("yolo12n.pt")
object_counter = {"car": set(), "motorcycle": set(), "truck": set()}
object_last_positions = {}
object_speeds = {}

## Tracking objects and Speed calculation

In [17]:
# Centroid tracking helper
def get_centroid(x1, y1, x2, y2):
    return ((x1 + x2) // 2, (y1 + y2) // 2)

def calculate_speed(prev, curr):
    dx = curr[0] - prev[0]
    dy = curr[1] - prev[1]
    dist_pixels = math.sqrt(dx**2 + dy**2)
    dist_feet = dist_pixels / math.sqrt(PIXELS_PER_FOOT_X * PIXELS_PER_FOOT_Y)
    speed_fps = dist_feet * FRAME_RATE
    speed_kmph = speed_fps * 1.09728
    return round(speed_kmph, 2)

## Frames, Height, Weights, id assigning and counting

In [ ]:
# Start processing
cap = cv2.VideoCapture(VIDEO_PATH)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter("output_diff.avi", cv2.VideoWriter_fourcc(*'XVID'), fps, (width, height))

In [ ]:
id_counter = 0
tracked_objects = {}  # obj_id: (cx, cy)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)[0]
    detections = []

    for det in results.boxes:
        cls_id = int(det.cls[0])
        cls_name = model.names[cls_id]
        if cls_name not in ["car", "motorcycle", "truck"]:
            continue

        x1, y1, x2, y2 = map(int, det.xyxy[0])
        cx, cy = get_centroid(x1, y1, x2, y2)
        matched_id = None

        # Simple centroid tracker (no Kalman, no Norfair)
        for obj_id, (prev_cx, prev_cy) in tracked_objects.items():
            if math.hypot(cx - prev_cx, cy - prev_cy) < 60:  # threshold
                matched_id = obj_id
                break

        if matched_id is None:
            matched_id = id_counter
            id_counter += 1

        tracked_objects[matched_id] = (cx, cy)

        # Counting logic (crossing the line)
        if matched_id not in object_counter[cls_name] and cy > COUNT_LINE_Y:
            object_counter[cls_name].add(matched_id)

        # Speed estimation
        if matched_id in object_last_positions:
            prev_pos = object_last_positions[matched_id]
            speed = calculate_speed(prev_pos, (cx, cy))
            object_speeds[matched_id] = speed
        object_last_positions[matched_id] = (cx, cy)

        speed = object_speeds.get(matched_id, 0)
        color = (0, 0, 255) if speed > SPEED_LIMIT_KMPH else (0, 255, 0)

        # Draw
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{cls_name} ID:{matched_id} {speed} km/h", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Draw count line
    cv2.line(frame, (0, COUNT_LINE_Y), (width, COUNT_LINE_Y), (255, 255, 0), 2)

    # Display counts
    y_offset = 30
    for cls in ["car", "motorcycle", "truck"]:
        cv2.putText(frame, f"{cls}: {len(object_counter[cls])}", (10, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        y_offset += 25

    out.write(frame)

cap.release()
out.release()

print("✅ Processing complete. Video saved as output.avi")

# Faster R-CNN Model

## Installing Libraries

In [1]:
!pip install torch torchvision opencv-python-headless numpy norfair

ERROR: Could not find a version that satisfies the requirement detectron2 (from versions: none)
ERROR: No matching distribution found for detectron2


## Importing Libraries

In [16]:
import cv2
import torch
import torchvision
import numpy as np
from torchvision.transforms import functional as F
from collections import defaultdict
import math

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## Model Selection

In [ ]:
# ====== Model ======
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

## COCO classes for ResNet Model

In [ ]:

# ====== Classes ======
COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag',
    'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite',
    'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana',
    'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table',
    'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock',
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

vehicle_classes = {'car', 'motorcycle', 'truck'}
vehicle_ids = [COCO_INSTANCE_CATEGORY_NAMES.index(cls) for cls in vehicle_classes]

## Configration

In [ ]:
# ====== CONFIG ======
video_path = "/kaggle/input/yolo-testing-traffic/traffic.mp4"
COUNT_LINE_Y = 500
SPEED_LIMIT = 30  # km/h
PIXELS_PER_FOOT_X = 8
PIXELS_PER_FOOT_Y = 20

## Speed Calculation

In [1]:
# ====== Speed Calculation ======
def estimate_speed(prev, curr, fps):
    dx = curr[0] - prev[0]
    dy = curr[1] - prev[1]
    pixel_dist = math.hypot(dx, dy)
    feet_dist = pixel_dist / math.sqrt(PIXELS_PER_FOOT_X * PIXELS_PER_FOOT_Y)
    speed_fps = feet_dist * fps
    return speed_fps * 1.09728  # fps to km/h

# ====== Init Video ======
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width, height = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter('fixed_count_line_output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

object_tracks = {}
object_counter = defaultdict(set)
frame_index = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    img_tensor = F.to_tensor(frame).to(device)

    with torch.no_grad():
        outputs = model([img_tensor])[0]

    for box, label, score in zip(outputs['boxes'], outputs['labels'], outputs['scores']):
        if score < 0.7 or label.item() not in vehicle_ids:
            continue

        cls_name = COCO_INSTANCE_CATEGORY_NAMES[label.item()]
        x1, y1, x2, y2 = box.int().tolist()
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        matched_id = None

        for oid, track in object_tracks.items():
            if frame_index - track['last_seen'] < 10:
                prev_cx, prev_cy = track['center']
                if math.hypot(cx - prev_cx, cy - prev_cy) < 50:
                    matched_id = oid
                    break

        if matched_id is None:
            matched_id = len(object_tracks) + 1

        track = object_tracks.get(matched_id, {})
        speed = 0
        if 'prev_center' in track:
            speed = estimate_speed(track['prev_center'], (cx, cy), fps)

        object_tracks[matched_id] = {
            'center': (cx, cy),
            'last_seen': frame_index,
            'prev_center': (cx, cy),
            'class': cls_name
        }

        # Count if object crosses the line
        if cy > COUNT_LINE_Y:
            object_counter[cls_name].add(matched_id)

        color = (0, 0, 255) if speed > SPEED_LIMIT else (0, 255, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{cls_name} ID:{matched_id} {speed:.1f} km/h", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Draw count line
    cv2.line(frame, (0, COUNT_LINE_Y), (width, COUNT_LINE_Y), (255, 255, 0), 2)

    # Show object counts
    y_offset = 30
    for cls in vehicle_classes:
        count = len(object_counter[cls])
        cv2.putText(frame, f"{cls}: {count}", (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        y_offset += 25

    out.write(frame)
    frame_index += 1

cap.release()
out.release()
print("✅ Done. Output saved as 'fixed_count_line_output.mp4'")


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ Done. Output saved as 'fixed_count_line_output.mp4'


## Saving the Model

In [2]:
torch.save(model.state_dict(), "fasterrcnn_vehicle_detector.pth")
print("✅ Model weights saved to fasterrcnn_vehicle_detector.pth")

✅ Model weights saved to fasterrcnn_vehicle_detector.pth
